# 🌿 Project 3: Leaf Disease Detection

---

## 🧠 1️⃣ Project Overview

Task: Detect diseases in plant leaves (e.g., Healthy, Rust, Blight, etc.)  
Dataset: Public datasets like PlantVillage (~50k images, multiple classes)  
Input: RGB leaf images (224x224 recommended for Transfer Learning)  
Output: Multi-class classification  
Challenge:  
- Varied lighting & background  
- Multiple disease types  

---

## 🔹 2️⃣ Preprocessing Images


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    'data/leaf_dataset',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    'data/leaf_dataset',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


✅ Notes

- categorical → multi-class classification  
- Data augmentation helps generalization  

---

## 🟢 3️⃣ Basic CNN Model


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

num_classes = train_generator.num_classes

model = Sequential(
    [
        Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ]
)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 🔵 4️⃣ Train the Model

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=20,
)

✅ Notes:

- With large datasets → more epochs needed (20–50)

- Consider EarlyStopping to prevent overfitting

## 🟣 5️⃣ Using Transfer Learning (Recommended)

- Instead of training from scratch, we can use pre-trained models like MobileNetV2, ResNet50, VGG16.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

base_model = MobileNetV2(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze base model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 🔹 6️⃣ Fine-tuning the Model

In [ ]:
# Unfreeze some top layers of base model
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Recompile & train
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(train_generator, validation_data=validation_generator, epochs=10)

✅ Fine-tuning improves accuracy significantly.

## 🔹 7️⃣ Predicting New Leaf Images

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img("data/leaf_dataset/test/leaf1.jpg", target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0) / 255.0

pred = model.predict(x)
class_index = np.argmax(pred)
class_label = list(train_generator.class_indices.keys())[class_index]
print("Predicted Class:", class_label)

## 🔹 8️⃣ Tips for Real-world Leaf Disease Detection

- Use Transfer Learning → saves training time  
- Data Augmentation → crucial for real-world images  
- Combine Dropout + BatchNorm → better generalization  
- Use Confusion Matrix → evaluate multi-class accuracy  

---

## ✅ Summary of CNN Projects

| Project       | Type        | Dataset       | Key Points                                      |
|---------------|------------|---------------|------------------------------------------------|
| Cat vs Dog    | Binary     | 25k images    | Augmentation, Dropout, CNN layers             |
| MNIST Digits  | Multi-class| 60k images    | Flatten, Conv layers, BatchNorm optional      |
| Leaf Disease  | Multi-class| 50k+ images   | Transfer Learning, Fine-tuning, Real-world robustness |


# Full Project

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.preprocessing import image
import numpy as np

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    "data/leaf_dataset",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training",
)

validation_generator = train_datagen.flow_from_directory(
    "data/leaf_dataset",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
)


num_classes = train_generator.num_classes

model = Sequential(
    [
        Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation="relu"),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ]
)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=20,
)


base_model = MobileNetV2(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze base model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# Unfreeze some top layers of base model
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Recompile & train
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(train_generator, validation_data=validation_generator, epochs=10)


img = image.load_img("data/leaf_dataset/test/leaf1.jpg", target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0) / 255.0

pred = model.predict(x)
class_index = np.argmax(pred)
class_label = list(train_generator.class_indices.keys())[class_index]
print("Predicted Class:", class_label)